# Gated Attention for LLMs — G1–G5 Ablation (Unfrozen)
Reproduces core findings of *Gated Attention for Large Language Models: Non-linearity, Sparsity, and Attention-Sink-Free* (arXiv 2505.06708) at small scale.

| | |
|---|---|
| **Model** | GPT-2 small (124M params, 12 layers, 12 heads) |
| **Dataset** | WikiText-103 |
| **Task** | Language modelling (perplexity) |
| **Backbone** | **Unfrozen** — full fine-tuning |
| **Gates** | G1 (post-SDPA), G2 (value), G3 (key), G4 (query), G5 (dense output) |
| **GPU** | T4 on Kaggle (~4–6 hrs total) |

**Key differences from frozen variant:**
- All GPT-2 weights train alongside gate params
- Lower LR (2e-5) to avoid catastrophic forgetting
- Gate params zero-init → start neutral (σ=0.5)
- Includes word-level attention heatmap for qualitative inspection

## 0. Setup

In [ ]:
import os, types, math, json, shutil, glob as _glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import pearsonr
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  TEST MODE                                                               ║
# ║                                                                          ║
# ║  TEST = True  → full end-to-end sanity run (~5 min on T4)               ║
# ║                 tiny data, 1 epoch, all 8 figures produced               ║
# ║                 use this FIRST to confirm the notebook runs clean        ║
# ║                                                                          ║
# ║  TEST = False → full training (~3–4 hrs on T4, publication quality)     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
TEST = True   # ← flip to False for the real run

# ── All hyperparameters auto-scale with TEST ──────────────────────────────
SEQ_LEN      = 512
BATCH_SIZE   = 4   if TEST else 8
EPOCHS       = 1   if TEST else 5      # total target epochs (resumes if partial ckpt exists)
LR           = 2e-5
GRAD_CLIP    = 1.0
TRAIN_TOKENS = 25_600  if TEST else 2_000_000
VAL_TOKENS   = 10_240  if TEST else None
TEST_TOKENS  = 10_240  if TEST else None
EVAL_BATCHES = 5       if TEST else None
SINK_BATCHES = 3       if TEST else 10
GATE_BATCHES = 5       if TEST else 20
DISPLAY_LEN  = 20      if TEST else 40
EVAL_EVERY   = 5       if TEST else 50

CKPT_DIR = Path('checkpoints_llm_unfrozen' + ('_test' if TEST else ''))
CKPT_DIR.mkdir(exist_ok=True)

def _safe_name(n):
    return n.replace(' ', '_').replace('—', '').replace('(', '').replace(')', '').replace('/', '')

print(f'torch {torch.__version__} | device {DEVICE}')
print(f'TEST = {TEST}')
print(f'  epochs={EPOCHS}  batch={BATCH_SIZE}  lr={LR}')
print(f'  train_tokens={TRAIN_TOKENS:,}  val_tokens={VAL_TOKENS}  eval_every={EVAL_EVERY}')
print(f'  display_len={DISPLAY_LEN}  sink_batches={SINK_BATCHES}  gate_batches={GATE_BATCHES}')
print(f'Checkpoints → {CKPT_DIR.resolve()}')

## 0b. Checkpoint Restore (Kaggle resume)

In [ ]:
# ── Kaggle checkpoint restore ─────────────────────────────────────────────
# Attach previous version output as input dataset on Kaggle.
# Path pattern: /kaggle/input/notebooks/<user>/<notebook>/checkpoints_llm_unfrozen/
#
# To attach: Notebook → Settings → Data → Add input → Your Work → select version

_ckpt_dir_name = CKPT_DIR.name  # 'checkpoints_llm_unfrozen' or 'checkpoints_llm_unfrozen_test'

_src_dirs = (
    _glob.glob(f'/kaggle/input/**/{_ckpt_dir_name}', recursive=True) +
    _glob.glob(f'/kaggle/input/notebooks/**/{_ckpt_dir_name}', recursive=True)
)
_src_dirs = sorted(set(_src_dirs))

if _src_dirs:
    _src = Path(_src_dirs[-1])
    print(f'Found checkpoint source: {_src}')
    _copied = 0
    for _f in _src.iterdir():
        _dst = CKPT_DIR / _f.name
        if not _dst.exists():
            shutil.copy2(_f, _dst)
            print(f'  Copied : {_f.name}  ({_f.stat().st_size/1e6:.1f} MB)')
            _copied += 1
        else:
            print(f'  Exists : {_f.name}')
    print(f'Done — {_copied} file(s) copied.')
else:
    print('No previous checkpoint source found — starting fresh.')

print('\nCheckpoints in working dir:')
for _f in sorted(CKPT_DIR.iterdir()):
    print(f'  {_f.name}  ({_f.stat().st_size/1e6:.1f} MB)')

## 1. Dataset — WikiText-103

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset('wikitext', 'wikitext-103-raw-v1')

def tokenize_and_chunk(split, max_tokens=None):
    """
    Tokenize WikiText-103 article-by-article to avoid materialising the full
    corpus as one giant string (which causes OOM on the 100M-token train set).
    Stops as soon as max_tokens have been collected.
    """
    ids_list = []
    collected = 0
    limit = max_tokens or int(1e18)

    for row in raw[split]:
        text = row['text'].strip()
        if not text:
            continue
        enc = tokenizer(text, add_special_tokens=False, return_tensors='pt')
        toks = enc['input_ids'][0]
        if collected + len(toks) > limit:
            toks = toks[:limit - collected]
        ids_list.append(toks)
        collected += len(toks)
        if collected >= limit:
            break

    all_ids = torch.cat(ids_list)          # (total_tokens,)
    n_chunks = len(all_ids) // SEQ_LEN
    all_ids  = all_ids[:n_chunks * SEQ_LEN].reshape(n_chunks, SEQ_LEN)
    return all_ids

print('Tokenising train ...')
train_ids = tokenize_and_chunk('train',      max_tokens=TRAIN_TOKENS)
print('Tokenising validation ...')
val_ids   = tokenize_and_chunk('validation', max_tokens=VAL_TOKENS)
print('Tokenising test ...')
test_ids  = tokenize_and_chunk('test',       max_tokens=TEST_TOKENS)

class TokenDataset(Dataset):
    def __init__(self, chunks): self.chunks = chunks
    def __len__(self):          return len(self.chunks)
    def __getitem__(self, i):   return self.chunks[i]

train_loader = DataLoader(TokenDataset(train_ids), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(TokenDataset(val_ids),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(TokenDataset(test_ids),  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train : {len(train_ids):>5} chunks  ({len(train_ids)*SEQ_LEN/1e3:.0f}k tokens)')
print(f'Val   : {len(val_ids):>5} chunks  ({len(val_ids)*SEQ_LEN/1e3:.0f}k tokens)')
print(f'Test  : {len(test_ids):>5} chunks  ({len(test_ids)*SEQ_LEN/1e3:.0f}k tokens)')
print(f'Steps/epoch : {len(train_loader)}')
if TEST:
    print('[TEST MODE] Small dataset — flip TEST=False for full run.')

## 2. Gate Modules — G1 through G5

| Gate | Position | Gate equation |
|------|----------|--------------|
| **G1** | After SDPA output (best in paper) | `attn_out *= σ(X @ W_θ)` |
| **G2** | After Value projection | `v *= σ(X @ W_θ)` |
| **G3** | After Key projection | `k *= σ(X @ W_θ)` |
| **G4** | After Query projection | `q *= σ(X @ W_θ)` |
| **G5** | After concat, before output proj | `out *= σ(X @ W_θ)` |

All gates are **head-specific** multiplicative sigmoid gates: `W_θ ∈ R^{d_model × n_heads}`

In [ ]:
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

class HeadGate(nn.Module):
    """Head-specific multiplicative sigmoid gate (paper Eq. 5)."""
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(d_model, n_heads))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N, d_model) → gate: (B, N, H)"""
        return torch.sigmoid(x @ self.W)


def _split_heads(tensor, n_head, head_size):
    return tensor.view(tensor.size()[:-1] + (n_head, head_size)).permute(0, 2, 1, 3)

def _merge_heads(tensor, n_head, head_size):
    return tensor.permute(0, 2, 1, 3).contiguous().view(
        tensor.size(0), tensor.size(2), n_head * head_size)

def _apply_gate(tensor_bhnd, gate_bnh):
    return tensor_bhnd * gate_bnh.permute(0, 2, 1).unsqueeze(-1)


def make_gated_forward(attn_module: GPT2Attention, gate: HeadGate, pos: str):
    """
    Patched GPT2Attention.forward compatible with transformers >= 4.40.
    Uses **kwargs to absorb any new arguments (cache_position, etc.)
    and handles both old (layer_past) and new (past_key_values) KV-cache API.
    """
    n_head    = attn_module.num_heads
    head_size = attn_module.head_dim
    embed_dim = attn_module.embed_dim

    def forward(self, hidden_states, **kwargs):
        B, N, _ = hidden_states.shape
        x = hidden_states
        g = gate(x)  # (B, N, H)

        # ── Resolve KV-cache arg (old API: layer_past, new API: past_key_values)
        layer_past        = kwargs.get('past_key_values', kwargs.get('layer_past', None))
        attention_mask    = kwargs.get('attention_mask', None)
        head_mask         = kwargs.get('head_mask', None)
        use_cache         = kwargs.get('use_cache', False)
        output_attentions = kwargs.get('output_attentions', False)

        # QKV
        qkv = self.c_attn(hidden_states)
        q, k, v = qkv.split(self.embed_dim, dim=2)
        q = _split_heads(q, n_head, head_size)
        k = _split_heads(k, n_head, head_size)
        v = _split_heads(v, n_head, head_size)

        # Pre-SDPA gates
        if pos == 'G4': q = _apply_gate(q, g)
        if pos == 'G3': k = _apply_gate(k, g)
        if pos == 'G2': v = _apply_gate(v, g)

        # KV cache
        if layer_past is not None:
            if isinstance(layer_past, tuple):
                k = torch.cat([layer_past[0], k], dim=-2)
                v = torch.cat([layer_past[1], v], dim=-2)
        present = (k, v) if use_cache else None

        # SDPA — manual path for attention capture, otherwise use flash
        if output_attentions or getattr(self, '_capture_attn', False):
            scale  = head_size ** -0.5
            scores = (q * scale) @ k.transpose(-2, -1)
            if attention_mask is not None:
                scores = scores + attention_mask
            if layer_past is None:
                Nq, Nk = q.size(-2), k.size(-2)
                cm = torch.triu(torch.ones(Nq, Nk, device=x.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None, None], float('-inf'))
            attn_w   = scores.softmax(-1)
            attn_w   = self.attn_dropout(attn_w)
            attn_out = attn_w @ v
            if getattr(self, '_capture_attn', False):
                self._last_attn = attn_w.detach()
        else:
            attn_out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=(layer_past is None),
            )
            attn_w = None

        # G1: post-SDPA gate
        if pos == 'G1':
            attn_out = _apply_gate(attn_out, g)
        # ── Capture gate & norm for ALL positions (Fig 5/6/7) ──
        if getattr(self, '_capture_gate', False):
            self._last_gate   = g.detach()
            self._last_x_norm = x.norm(dim=-1).detach()

        attn_out = _merge_heads(attn_out, n_head, head_size)

        # G5: post-concat gate
        if pos == 'G5':
            attn_out = (
                attn_out.view(B, N, n_head, head_size) * g.unsqueeze(-1)
            ).view(B, N, embed_dim)

        attn_out = self.c_proj(attn_out)
        attn_out = self.resid_dropout(attn_out)

        outputs = (attn_out, present)
        if output_attentions:
            outputs += (attn_w,)
        return outputs

    return types.MethodType(forward, attn_module)


print('Gate modules ready (G1/G2/G3/G4/G5) — kwargs-based signature.')

## 3. Model Builder — Unfrozen Backbone

In [ ]:
def inject_gates_gpt2(model: GPT2LMHeadModel, pos: str):
    gate_list = []
    d_model = model.config.n_embd
    n_heads = model.config.n_head
    for block in model.transformer.h:
        g = HeadGate(d_model, n_heads)
        block.attn.forward = make_gated_forward(block.attn, g, pos)
        gate_list.append(g)
    model.gate_params = nn.ModuleList(gate_list)
    return model


def build_model(pos='baseline'):
    """
    Build GPT-2 small with UNFROZEN backbone.
    All parameters train (gate_params + transformer + lm_head).
    """
    m = GPT2LMHeadModel.from_pretrained('gpt2')
    if pos != 'baseline':
        inject_gates_gpt2(m, pos)
    for p in m.parameters():
        p.requires_grad_(True)
    m = m.to(DEVICE)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in m.parameters())
    gate_extra = sum(
        p.numel() for p in m.gate_params.parameters()
    ) if hasattr(m, 'gate_params') else 0
    print(f'  [{pos}]  trainable {n_train:,} / {n_total:,}'
          + (f'  (+{gate_extra:,} gate params)' if gate_extra else ''))
    return m


print('build_model() ready — no sanity-check build to save VRAM.')

## 4. Evaluation — Perplexity

In [ ]:
@torch.no_grad()
def evaluate_ppl(model, loader, max_batches=EVAL_BATCHES):
    """Compute perplexity. max_batches=None uses full loader."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        ids = batch.to(DEVICE)
        B, N = ids.shape
        out = model(ids, labels=ids)
        total_loss   += out.loss.item() * B * (N - 1)
        total_tokens += B * (N - 1)
    return math.exp(total_loss / total_tokens)

print('evaluate_ppl ready.')

## 5. Training Loop

In [ ]:
RESULTS_FILE = CKPT_DIR / 'results.json'


def train_one(model, name='model', epochs=EPOCHS, lr=LR,
              ckpt_path=None, resume_ckpt=None):
    """
    Train for `epochs` total epochs.
    If ckpt_path already exists and contains 'epochs_done', resumes from there.
    Passing resume_ckpt (an epoch-level .pth) overrides for mid-epoch recovery.
    """
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01
    )
    total_steps = epochs * len(train_loader)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(total_steps, 1))

    log = {'step': [], 'train_loss': [], 'val_ppl': []}
    start_epoch, step = 0, 0

    # ── Priority 1: resume from mid-epoch epoch-level checkpoint ──────────
    if resume_ckpt and Path(resume_ckpt).exists():
        ckpt = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['state'])
        opt.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        log        = ckpt['log']
        start_epoch = ckpt['epoch']
        step        = start_epoch * len(train_loader)
        print(f'  Resumed from epoch-ckpt {Path(resume_ckpt).name}  (epoch {start_epoch})')

    # ── Priority 2: resume from final checkpoint (already has some epochs) ─
    elif ckpt_path and Path(ckpt_path).exists():
        data = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(data['state'])
        log         = data.get('log', log)
        start_epoch = data.get('epochs_done', 0)
        step        = start_epoch * len(train_loader)
        if start_epoch > 0:
            print(f'  Loaded final ckpt — already {start_epoch} epoch(s) done.')
            if start_epoch >= epochs:
                print(f'  Target {epochs} epochs already reached — skipping training.')
                return log

    if start_epoch >= epochs:
        print(f'  Nothing to train (start_epoch={start_epoch} >= epochs={epochs})')
        return log

    print(f'  Training epochs {start_epoch+1}–{epochs}  (lr={lr}, batch={BATCH_SIZE})')

    for ep in range(start_epoch, epochs):
        model.train()
        ep_loss = 0.0
        pbar = tqdm(train_loader, desc=f'{name} ep{ep+1}/{epochs}', leave=False)
        for batch in pbar:
            ids = batch.to(DEVICE)
            opt.zero_grad()
            loss = model(ids, labels=ids).loss
            loss.backward()
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP
            )
            opt.step()
            sched.step()
            ep_loss += loss.item()
            step    += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')

            if step % EVAL_EVERY == 0:
                vp = evaluate_ppl(model, val_loader)
                log['step'].append(step)
                log['train_loss'].append(loss.item())
                log['val_ppl'].append(vp)
                model.train()

        avg = ep_loss / len(train_loader)
        vp  = evaluate_ppl(model, val_loader)
        print(f'  ep{ep+1}  avg_loss={avg:.4f}  val_ppl={vp:.2f}')

        # Save epoch checkpoint (crash recovery)
        sname   = _safe_name(name)
        ep_ckpt = CKPT_DIR / f'{sname}_epoch{ep+1}.pth'
        torch.save({
            'epoch': ep + 1, 'state': model.state_dict(),
            'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(),
            'log': log,
        }, ep_ckpt)
        prev = CKPT_DIR / f'{sname}_epoch{ep}.pth'
        if prev.exists(): prev.unlink()

    # Save final checkpoint — includes epochs_done for future resume
    if ckpt_path:
        torch.save({'state': model.state_dict(), 'log': log, 'epochs_done': epochs}, ckpt_path)
        print(f'  saved → {ckpt_path}  (epochs_done={epochs})')
    return log

print('Training loop ready.')

## 6. Run Ablation — Baseline + G1–G5
Edit `EXPERIMENTS` to add/remove variants. Auto-skips if checkpoint exists.

In [ ]:
# ── Tag old checkpoints that predate the epochs_done field ────────────────
# If you previously ran N epochs without this field, set PREVIOUS_EPOCHS_DONE
# so the resume logic knows where to continue from.
#
# Example: ran 3 epochs before → set PREVIOUS_EPOCHS_DONE = 3
#          first run ever      → leave as 0 (no-op, nothing to tag)

PREVIOUS_EPOCHS_DONE = 0   # ← change to 3 if continuing from a 3-epoch run

if PREVIOUS_EPOCHS_DONE > 0:
    _all_names = ['Baseline', 'G1__SDPA_output', 'G2__Value_proj',
                  'G3__Key_proj', 'G4__Query_proj', 'G5__Dense_output']
    _tagged = 0
    for _sname in _all_names:
        _ckpt = CKPT_DIR / f'gpt2_{_sname}.pth'
        if _ckpt.exists():
            _d = torch.load(_ckpt, map_location='cpu', weights_only=False)
            if 'epochs_done' not in _d:
                _d['epochs_done'] = PREVIOUS_EPOCHS_DONE
                torch.save(_d, _ckpt)
                print(f'  Tagged {_ckpt.name} → epochs_done={PREVIOUS_EPOCHS_DONE}')
                _tagged += 1
            else:
                print(f'  Already tagged: {_ckpt.name}  epochs_done={_d["epochs_done"]}')
    print(f'Tagged {_tagged} checkpoint(s).')
else:
    print('PREVIOUS_EPOCHS_DONE=0 — no tagging needed (fresh run).')

In [ ]:
EXPERIMENTS = [
    ('Baseline',          'baseline'),
    ('G1 — SDPA output',  'G1'),
    ('G2 — Value proj',   'G2'),
    ('G3 — Key proj',     'G3'),
    ('G4 — Query proj',   'G4'),
    ('G5 — Dense output', 'G5'),
]

# Load previously saved results if any
if RESULTS_FILE.exists():
    with open(RESULTS_FILE) as f:
        results_saved = json.load(f)
    print(f'Resumed {len(results_saved)} saved result(s) from {RESULTS_FILE}')
else:
    results_saved = {}

results = {}   # name → {log, test_ppl}

for name, pos in EXPERIMENTS:
    sname = _safe_name(name)
    ckpt  = CKPT_DIR / f'gpt2_{sname}.pth'

    print(f'\n── {name} ──')

    # Check if final ckpt exists and is already at target epochs
    if ckpt.exists():
        _meta = torch.load(ckpt, map_location='cpu', weights_only=False)
        epochs_done = _meta.get('epochs_done', 0)
        if epochs_done >= EPOCHS and name in results_saved:
            print(f'  [SKIP] {epochs_done} epochs done, results saved')
            results[name] = {'log': _meta['log'], 'test_ppl': results_saved[name]['test_ppl']}
            continue
        else:
            print(f'  Found ckpt with {epochs_done} epoch(s) — will train to {EPOCHS}')

    # Build model (train_one will load weights from ckpt if it exists)
    model = build_model(pos)

    # Check for mid-epoch crash recovery checkpoint
    epoch_ckpts = sorted(_glob.glob(str(CKPT_DIR / f'{sname}_epoch*.pth')))
    resume = epoch_ckpts[-1] if epoch_ckpts else None
    if resume:
        print(f'  Mid-epoch ckpt found: {Path(resume).name}')

    log = train_one(model, name=name, ckpt_path=ckpt, resume_ckpt=resume)

    test_ppl = evaluate_ppl(model, test_loader)
    print(f'  test PPL = {test_ppl:.2f}')

    results[name] = {'log': log, 'test_ppl': test_ppl}
    results_saved[name] = {'test_ppl': test_ppl}
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results_saved, f, indent=2)

    # FREE VRAM before next variant
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f'\nAll {len(EXPERIMENTS)} experiments done.')
print(f'Peak VRAM never holds more than 1 model at a time.')

## 7. Analysis Helpers — Load Models One at a Time
All analysis cells load one model, compute, then immediately free it to avoid OOM.

In [ ]:
def load_for_analysis(name: str, pos: str) -> GPT2LMHeadModel:
    """
    Load a trained model for analysis.
    Always call  del model; torch.cuda.empty_cache()  after use.
    """
    sname = _safe_name(name)
    ckpt  = CKPT_DIR / f'gpt2_{sname}.pth'
    if not ckpt.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt}')
    m = build_model(pos)
    data = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    m.load_state_dict(data['state'])
    m.eval()
    return m

print('load_for_analysis() ready.')

## 8. Attention Sink Analysis

In [ ]:
def get_first_token_attn(model, loader, n_batches=SINK_BATCHES):
    """Per-layer mean attention to first token (attention-sink proxy)."""
    n_layers = model.config.n_layer
    n_heads  = model.config.n_head
    head_dim = model.config.n_embd // n_heads
    all_first = [[] for _ in range(n_layers)]
    handles   = []

    for li, block in enumerate(model.transformer.h):
        cap = {}
        def make_hook(idx, c):
            def hook(module, inp, out):
                hs = inp[0]
                B, N, C = hs.shape
                qkv = module.c_attn(hs)
                q, k, _ = qkv.split(module.embed_dim, dim=2)
                q = _split_heads(q, n_heads, head_dim)
                k = _split_heads(k, n_heads, head_dim)
                scores = (q * (head_dim ** -0.5)) @ k.transpose(-2, -1)
                cm = torch.triu(torch.ones(N, N, device=hs.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None, None], float('-inf'))
                w = scores.softmax(-1)
                c['first'] = w[:, :, :, 0].mean(dim=(1, 2)).detach()
            return hook
        h = block.attn.register_forward_hook(make_hook(li, cap))
        handles.append((li, cap, h))

    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            model(batch.to(DEVICE))
            for li, cap, _ in handles:
                if 'first' in cap:
                    all_first[li].append(cap['first'].mean().item())

    for _, _, h in handles: h.remove()
    return [np.mean(v) if v else 0.0 for v in all_first]


# Load Baseline → compute → free
print('Computing attention-sink (Baseline)...')
_m = load_for_analysis('Baseline', 'baseline')
sink_baseline = get_first_token_attn(_m, val_loader)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()

# Load G1 → compute → free
print('Computing attention-sink (G1)...')
_m = load_for_analysis('G1 — SDPA output', 'G1')
sink_g1 = get_first_token_attn(_m, val_loader)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'Baseline  mean first-token attn: {np.mean(sink_baseline):.4f}')
print(f'G1        mean first-token attn: {np.mean(sink_g1):.4f}')
print(f'Reduction: {(1 - np.mean(sink_g1)/np.mean(sink_baseline))*100:.1f}%')

## 9. Gate Sparsity & Gate-vs-Norm Analysis

In [ ]:
@torch.no_grad()
def collect_gate_scores(model, loader, n_batches=GATE_BATCHES):
    """Mean gate score per layer for a G1 model."""
    for block in model.transformer.h:
        block.attn._capture_gate = True
    all_gates = [[] for _ in range(model.config.n_layer)]
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        for li, block in enumerate(model.transformer.h):
            if hasattr(block.attn, '_last_gate'):
                all_gates[li].append(block.attn._last_gate.mean().item())
    for block in model.transformer.h:
        block.attn._capture_gate = False
    return [np.mean(v) if v else 0.0 for v in all_gates]


@torch.no_grad()
def collect_gate_vs_norm(model, loader, n_batches=GATE_BATCHES):
    """Collect gate scores vs token norms from the last layer."""
    last_block = model.transformer.h[-1]
    last_block.attn._capture_gate = True
    all_norms, all_gates = [], []
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        if hasattr(last_block.attn, '_last_gate'):
            all_gates.append(last_block.attn._last_gate.mean(-1).cpu().flatten().numpy())
            all_norms.append(last_block.attn._last_x_norm.cpu().flatten().numpy())
    last_block.attn._capture_gate = False
    if not all_gates:
        return np.array([]), np.array([])
    return np.concatenate(all_norms), np.concatenate(all_gates)


# Load G1 → compute gate stats → free
print('Computing gate sparsity & gate-vs-norm (G1)...')
_m = load_for_analysis('G1 — SDPA output', 'G1')
gate_scores_per_layer = collect_gate_scores(_m, val_loader)
norms_all, gates_all  = collect_gate_vs_norm(_m, val_loader)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()

print('G1 mean gate per layer:', [f'{s:.3f}' for s in gate_scores_per_layer])

if len(norms_all) > 0:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(norms_all), min(5000, len(norms_all)), replace=False)
    norms_s, gates_s = norms_all[idx], gates_all[idx]
    r_gn, p_gn = pearsonr(norms_s, gates_s)
    print(f'Gate vs norm Pearson r={r_gn:.3f}  p={p_gn:.2e}  n={len(norms_s)}')
else:
    r_gn, p_gn = float('nan'), float('nan')
    norms_s, gates_s = np.array([]), np.array([])
    print('No gate data captured (baseline has no gates — expected).')

## 10. Word-Level Attention Matrices

In [ ]:
# ── Select a readable sample sentence from the validation set ─────────────
for ci in range(len(val_ids)):
    text = tokenizer.decode(val_ids[ci].tolist(), skip_special_tokens=True)
    if len(text.strip()) > 80 and '=' not in text[:30]:
        SAMPLE_CHUNK_IDX = ci
        break

SAMPLE_IDS = val_ids[SAMPLE_CHUNK_IDX].unsqueeze(0)  # (1, SEQ_LEN)
sample_tokens  = tokenizer.convert_ids_to_tokens(SAMPLE_IDS[0, :DISPLAY_LEN].tolist())
display_tokens = [t.replace('\u0120', ' ').replace('\u010a', '\\n') for t in sample_tokens]

print(f'Sample [{SAMPLE_CHUNK_IDX}] first {DISPLAY_LEN} tokens:')
print(' '.join(display_tokens))


def get_attn_matrix(model, input_ids, n_tokens=DISPLAY_LEN, is_baseline=False):
    """
    Mean attention matrix (n_tokens × n_tokens) over all layers & heads.

    For baseline: temporarily disables fused/flash attention so that
    output_attentions=True actually returns weights (flash attn discards them).
    For gated models: uses _capture_attn flag in the patched forward.
    """
    model.eval()

    if is_baseline:
        # Disable fused attn on all attention modules so manual path runs
        for block in model.transformer.h:
            block.attn.is_causal = True   # ensure causal mask
            if hasattr(block.attn, '_attn_implementation'):
                block.attn._attn_implementation = 'eager'
        # Patch config too (newer transformers reads this)
        _orig_impl = getattr(model.config, '_attn_implementation', 'sdpa')
        model.config._attn_implementation = 'eager'

        with torch.no_grad():
            out = model(input_ids.to(DEVICE), output_attentions=True)

        # Restore
        model.config._attn_implementation = _orig_impl

        if out.attentions is None:
            # Final fallback: manually compute via forward hooks
            print('  output_attentions still None — using hook fallback')
            return _get_attn_via_hook(model, input_ids, n_tokens)

        attn_layers = [a[0].cpu().numpy() for a in out.attentions]  # each (H, N, N)

    else:
        # Patched forward path — uses _capture_attn flag
        for block in model.transformer.h:
            block.attn._capture_attn = True
        with torch.no_grad():
            model(input_ids.to(DEVICE))
        attn_layers = []
        for block in model.transformer.h:
            if hasattr(block.attn, '_last_attn'):
                attn_layers.append(block.attn._last_attn[0].cpu().numpy())
            block.attn._capture_attn = False

    if not attn_layers:
        print('  WARNING: no attention captured — returning zeros')
        return np.zeros((n_tokens, n_tokens))

    mat = np.stack(attn_layers).mean(axis=(0, 1))  # (N, N)
    return mat[:n_tokens, :n_tokens]


def _get_attn_via_hook(model, input_ids, n_tokens=DISPLAY_LEN):
    """Hook-based fallback for baseline: manually computes softmax attention."""
    n_heads  = model.config.n_head
    head_dim = model.config.n_embd // n_heads
    attn_layers = []
    handles = []

    for block in model.transformer.h:
        cap = {}
        def make_hook(c):
            def hook(module, inp, out):
                hs = inp[0]; B, N, _ = hs.shape
                qkv = module.c_attn(hs)
                q, k, _ = qkv.split(module.embed_dim, dim=2)
                q = q.view(B, N, n_heads, head_dim).permute(0,2,1,3)
                k = k.view(B, N, n_heads, head_dim).permute(0,2,1,3)
                scores = (q * head_dim**-0.5) @ k.transpose(-2,-1)
                cm = torch.triu(torch.ones(N, N, device=hs.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None,None], float('-inf'))
                c['attn'] = scores.softmax(-1)[0].detach().cpu().numpy()  # (H, N, N)
            return hook
        h = block.attn.register_forward_hook(make_hook(cap))
        handles.append((cap, h))

    model.eval()
    with torch.no_grad():
        model(input_ids.to(DEVICE))

    for cap, h in handles:
        h.remove()
        if 'attn' in cap:
            attn_layers.append(cap['attn'])

    if not attn_layers:
        return np.zeros((n_tokens, n_tokens))
    mat = np.stack(attn_layers).mean(axis=(0, 1))
    return mat[:n_tokens, :n_tokens]


# Load each model, extract attn matrix, immediately free
print('Computing attention matrices...')

_m = load_for_analysis('Baseline', 'baseline')
attn_baseline = get_attn_matrix(_m, SAMPLE_IDS, is_baseline=True)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()
print(f'  Baseline  max={attn_baseline.max():.4f}  min={attn_baseline.min():.4f}')

_m = load_for_analysis('G1 — SDPA output', 'G1')
attn_g1 = get_attn_matrix(_m, SAMPLE_IDS, is_baseline=False)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()
print(f'  G1        max={attn_g1.max():.4f}  min={attn_g1.min():.4f}')

_m = load_for_analysis('G5 — Dense output', 'G5')
attn_g5 = get_attn_matrix(_m, SAMPLE_IDS, is_baseline=False)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()
print(f'  G5        max={attn_g5.max():.4f}  min={attn_g5.min():.4f}')

print(f'Attention matrix shape: {attn_baseline.shape}')

## 11. Figures

In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.05)
names         = [n for n, _ in EXPERIMENTS]
ppls          = [results[n]['test_ppl'] for n in names]
baseline_ppl  = results['Baseline']['test_ppl']
deltas        = [baseline_ppl - p for p in ppls]

palette_var = {
    'Baseline':          '#888888',
    'G1 — SDPA output':  '#4C72B0',
    'G2 — Value proj':   '#55A868',
    'G3 — Key proj':     '#C44E52',
    'G4 — Query proj':   '#DD8452',
    'G5 — Dense output': '#937860',
}

In [ ]:
# ── Fig A — PPL ablation bar chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig A  |  Gate Position Ablation — Test PPL\n'
             '(GPT-2 small, WikiText-103, unfrozen backbone)',
             fontsize=13, fontweight='bold')

colors = [palette_var[n] for n in names]

ax = axes[0]
bars = ax.bar(names, ppls, color=colors, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars, ppls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.axhline(baseline_ppl, color='gray', linestyle='--', lw=1.2, label='Baseline')
ax.set_ylabel('Test Perplexity (↓ better)')
ax.set_title('Absolute Test PPL', fontweight='bold')
ax.set_xticklabels(names, rotation=25, ha='right')
ax.set_ylim(min(ppls)*0.97, max(ppls)*1.01)
ax.grid(axis='y', alpha=0.3)
ax.legend()

ax2 = axes[1]
dcol = ['#2ca02c' if d > 0 else ('#d62728' if d < 0 else '#888888') for d in deltas]
bars2 = ax2.bar(names, deltas, color=dcol, edgecolor='k', linewidth=0.6)
for bar, d in zip(bars2, deltas):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + (0.05 if d >= 0 else -0.25),
             f'{d:+.2f}', ha='center', va='bottom', fontsize=8)
ax2.axhline(0, color='k', lw=1)
ax2.set_ylabel('ΔPPL vs Baseline (↑ = better)')
ax2.set_title('PPL Improvement over Baseline', fontweight='bold')
ax2.set_xticklabels(names, rotation=25, ha='right')
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig('figA_ppl_ablation.pdf', bbox_inches='tight')
plt.show()
print('Saved figA_ppl_ablation.pdf')

In [ ]:
# ── Fig B — Training curves ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig B  |  Training Dynamics — GPT-2 small, WikiText-103 (Unfrozen)',
             fontsize=13, fontweight='bold')

for name, _ in EXPERIMENTS:
    log = results[name]['log']
    if log['step']:
        ax1.plot(log['step'], log['train_loss'], color=palette_var[name], label=name, alpha=0.85)
        ax2.plot(log['step'], log['val_ppl'],    color=palette_var[name], label=name, alpha=0.85)

ax1.set_xlabel('Step'); ax1.set_ylabel('Train Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

ax2.set_xlabel('Step'); ax2.set_ylabel('Val PPL')
ax2.set_title('Validation Perplexity', fontweight='bold')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig('figB_training_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved figB_training_curves.pdf')

In [ ]:
# ── Fig C — Attention Sink ───────────────────────────────────────────────────
layers_x = list(range(1, len(sink_baseline) + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig C  |  Attention Sink — First-Token Attention Score\n'
             'GPT-2 small, WikiText-103 val (Unfrozen)',
             fontsize=13, fontweight='bold')

ax1.plot(layers_x, sink_baseline, 'o-', color='#888888',
         label=f'Baseline  (mean={np.mean(sink_baseline):.3f})')
ax1.plot(layers_x, sink_g1,       's-', color='#4C72B0',
         label=f'G1 Gate   (mean={np.mean(sink_g1):.3f})')
ax1.axhline(np.mean(sink_baseline), color='#888888', linestyle='--', lw=1)
ax1.axhline(np.mean(sink_g1),       color='#4C72B0', linestyle='--', lw=1)
ax1.set_xlabel('Layer'); ax1.set_ylabel('Mean Attention to First Token')
ax1.set_title('Per-Layer Attention Sink', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(['Baseline', 'G1 Gate'],
        [np.mean(sink_baseline), np.mean(sink_g1)],
        color=['#888888', '#4C72B0'], edgecolor='k')
for i, v in enumerate([np.mean(sink_baseline), np.mean(sink_g1)]):
    ax2.text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('Mean First-Token Attention Score')
red_pct = (1 - np.mean(sink_g1)/np.mean(sink_baseline)) * 100
ax2.set_title(f'Attention Sink Reduction\n(G1 reduces sink by {red_pct:.1f}%)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig('figC_attention_sink.pdf', bbox_inches='tight')
plt.show()
print('Saved figC_attention_sink.pdf')

In [ ]:
# ── Fig D — Gate Sparsity per Layer (G1) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1, len(gate_scores_per_layer)+1), gate_scores_per_layer,
       color='#4C72B0', edgecolor='k', linewidth=0.5)
ax.axhline(0.5, color='red', linestyle='--', lw=1.2, label='σ=0.5 (neutral gate)')
ax.set_xlabel('Layer'); ax.set_ylabel('Mean Gate Score')
ax.set_title('Fig D  |  G1 Gate Sparsity per Layer\n'
             'Scores < 0.5 → active suppression of SDPA outputs (Unfrozen)',
             fontweight='bold')
ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('figD_gate_sparsity.pdf', bbox_inches='tight')
plt.show()
print('Saved figD_gate_sparsity.pdf')

In [ ]:
# ── Fig E — Gate Suppression vs Token Norm (scatter) ────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
if len(norms_s) > 0:
    ax.scatter(norms_s, gates_s, alpha=0.15, s=6, color='#4C72B0', rasterized=True)
    # Trend line
    z = np.polyfit(norms_s, gates_s, 1)
    xr = np.linspace(norms_s.min(), norms_s.max(), 200)
    ax.plot(xr, np.polyval(z, xr), 'r-', lw=2, label=f'r={r_gn:.3f}')
    ax.legend(fontsize=11)
else:
    ax.text(0.5, 0.5, 'No data (model has no gates)', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
ax.set_xlabel('Token L2 Norm  ||x||₂', fontsize=12)
ax.set_ylabel('Gate Score  σ(x·Wθ)', fontsize=12)
ax.set_title('Fig E  |  Gate Suppression vs Token Norm — G1, Last Layer\n'
             '(GPT-2 small, WikiText-103 val, Unfrozen)',
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('figE_gate_vs_norm.pdf', bbox_inches='tight')
plt.show()
print('Saved figE_gate_vs_norm.pdf')

In [ ]:
# ── Fig F — Layer × Head Gate Activation Heatmap (G1) ───────────────────────
print('Computing gate heatmap (G1)...')
_m = load_for_analysis('G1 — SDPA output', 'G1')
for block in _m.transformer.h:
    block.attn._capture_gate = True

n_layers    = _m.config.n_layer
n_heads     = _m.config.n_head
layer_head  = np.zeros((n_layers, n_heads))
n_collected = 0

_m.eval()
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= GATE_BATCHES: break
        _m(batch.to(DEVICE))
        for li, block in enumerate(_m.transformer.h):
            if hasattr(block.attn, '_last_gate'):
                layer_head[li] += block.attn._last_gate.mean(dim=(0, 1)).cpu().numpy()
        n_collected += 1

layer_head /= max(n_collected, 1)
del _m
if DEVICE == 'cuda': torch.cuda.empty_cache()

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    layer_head, annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=[f'H{i+1}' for i in range(n_heads)],
    yticklabels=[f'L{i+1}' for i in range(n_layers)],
    vmin=0, vmax=1, ax=ax, linewidths=0.4,
)
ax.set_xlabel('Attention Head', fontsize=12)
ax.set_ylabel('Transformer Layer', fontsize=12)
ax.set_title('Fig F  |  G1 Gate Activations — Mean over Batch & Tokens\n'
             '(Layer × Head)  GPT-2 small, WikiText-103 val (Unfrozen)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('figF_gate_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved figF_gate_heatmap.pdf')

In [ ]:
# ── Fig G — Word-Level Attention Heatmap ─────────────────────────────────────
VARIANTS_ATTN = [
    ('Baseline',          attn_baseline),
    ('G1 — SDPA output',  attn_g1),
    ('G5 — Dense output', attn_g5),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle(
    'Fig G  |  Word-Level Attention Heatmap\n'
    'Mean over all layers & heads · GPT-2 small · WikiText-103 val · Unfrozen',
    fontsize=13, fontweight='bold'
)

for ax, (vname, mat) in zip(axes, VARIANTS_ATTN):
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    sns.heatmap(
        mat,
        ax=ax,
        mask=mask,
        cmap='YlOrRd',
        xticklabels=display_tokens,
        yticklabels=display_tokens,
        cbar=True,
        linewidths=0.0,
        vmin=0,
    )
    ax.set_title(vname, fontsize=12, fontweight='bold')
    ax.set_xlabel('Key token (attended to)', fontsize=9)
    ax.set_ylabel('Query token (attending from)', fontsize=9)
    ax.set_xticklabels(display_tokens, rotation=90, fontsize=6)
    ax.set_yticklabels(display_tokens, rotation=0, fontsize=6)

fig.tight_layout()
fig.savefig('figG_word_attention_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved figG_word_attention_heatmap.pdf')
print('Row = query token (attending FROM), Col = key token (attending TO)')
print('Causal: each token can only attend to past tokens (upper triangle masked)')

In [ ]:
# ── Fig H — Per-token attention received (column-sum of attn matrix) ─────────
fig, axes = plt.subplots(3, 1, figsize=(18, 9))
fig.suptitle(
    'Fig H  |  Per-Token Attention Received (Column Sum)\n'
    'How much each word is attended to — averaged over all query positions & heads & layers',
    fontsize=13, fontweight='bold'
)

for ax, (vname, mat) in zip(axes, VARIANTS_ATTN):
    col_sum = mat.sum(axis=0)
    col_sum = col_sum / col_sum.sum()
    bar_colors = plt.cm.YlOrRd(col_sum / col_sum.max())
    ax.bar(range(len(display_tokens)), col_sum, color=bar_colors, edgecolor='none')
    ax.set_xticks(range(len(display_tokens)))
    ax.set_xticklabels(display_tokens, rotation=70, ha='right', fontsize=7)
    ax.set_ylabel('Attention weight')
    ax.set_title(vname, fontweight='bold', fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    top3 = np.argsort(col_sum)[-3:][::-1]
    for t in top3:
        ax.text(t, col_sum[t] + 0.002, '▲', ha='center', va='bottom', fontsize=8, color='red')

fig.tight_layout()
fig.savefig('figH_per_token_attn.pdf', bbox_inches='tight')
plt.show()
print('Saved figH_per_token_attn.pdf')
print('Red ▲ marks the top-3 most attended-to tokens per model')

## 12. Results Summary

In [ ]:
print('=' * 70)
print('  RESULTS SUMMARY')
print('  Model  : GPT-2 small (124M) — UNFROZEN backbone')
print(f'  Data   : WikiText-103 ({TRAIN_TOKENS/1e6:.1f}M token train slice)')
print(f'  Epochs : {EPOCHS}  |  LR: {LR}')
print('=' * 70)
print(f'  {"Method":<26} {"Test PPL":>10} {"ΔPPL":>8}')
print('  ' + '-' * 48)
best_ppl = min(ppls)
for name, ppl, delta in zip(names, ppls, deltas):
    tag = '  ← best' if ppl == best_ppl else ''
    print(f'  {name:<26} {ppl:>10.2f} {delta:>+8.2f}{tag}')

print()
print('  Attention Sink (first-token attn):')
print(f'    Baseline : {np.mean(sink_baseline):.4f}')
print(f'    G1 Gate  : {np.mean(sink_g1):.4f}')
print(f'    Reduction: {(1 - np.mean(sink_g1)/np.mean(sink_baseline))*100:.1f}%')
print()
print('  Gate-Norm correlation (G1, last layer):')
print(f'    Pearson r = {r_gn:.3f}  (p = {p_gn:.2e})')
print()
print('  Figures saved:')
figs = [
    ('figA_ppl_ablation.pdf',          'PPL bar chart — absolute & delta'),
    ('figB_training_curves.pdf',       'Training loss & val PPL curves'),
    ('figC_attention_sink.pdf',        'Attention sink per layer (Baseline vs G1)'),
    ('figD_gate_sparsity.pdf',         'G1 gate scores per layer'),
    ('figE_gate_vs_norm.pdf',          'Gate suppression vs token norm scatter'),
    ('figF_gate_heatmap.pdf',          'Layer x Head gate activation heatmap'),
    ('figG_word_attention_heatmap.pdf','Word-level attention N×N heatmap'),
    ('figH_per_token_attn.pdf',        'Per-token attention received bar chart'),
]
for fname, desc in figs:
    ok = '✓' if Path(fname).exists() else '✗'
    print(f'    {ok} {fname:<42} {desc}')
print('=' * 70)